# 02 — Estrategia de búsqueda reproducible

Este notebook construye consultas multilingües para la revisión integral sobre cambio climático y pesquerías. No ejecuta búsquedas externas automáticamente. Su función es generar, documentar y auditar las consultas antes de aplicarlas en cada base de datos o sitio institucional.

Se generan dos niveles:

1. **búsqueda amplia**: clima/variabilidad + pesca;
2. **búsqueda prioritaria**: clima/variabilidad + pesca + pequeños pelágicos y especies relacionadas.

In [ ]:
from pathlib import Path
import shutil
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from evidence_review.search import load_search_strategy, build_standard_queries

print(f'Project root: {ROOT}')

## 1. Cargar la estrategia

In [ ]:
strategy_path = ROOT / 'config' / 'search_strategy.yml'
strategy = load_search_strategy(strategy_path)

print(strategy['project']['review_title'])
print('Languages:', ', '.join(strategy['eligibility']['languages']))
print('Date from:', strategy['eligibility']['date_from'])

## 2. Generar consultas controladas

In [ ]:
queries = build_standard_queries(strategy)
queries_df = pd.DataFrame([
    {
        'language': item.language,
        'query_type': item.query_type,
        'blocks': ' + '.join(item.blocks),
        'exact_query': item.query,
    }
    for item in queries
])

queries_df[['language', 'query_type', 'blocks']]

## 3. Inspeccionar una consulta completa

Antes de usar una consulta en Scopus, Web of Science, OpenAlex u otra plataforma, revisa su sintaxis. Cada plataforma puede requerir adaptar campos, comodines o límites de longitud. La consulta finalmente ejecutada debe registrarse exactamente, no solo la versión conceptual.

In [ ]:
selected_language = 'en'
selected_type = 'priority_taxa'

selected = queries_df.query(
    'language == @selected_language and query_type == @selected_type'
).iloc[0]

print(selected['exact_query'])

## 4. Crear el registro local de búsquedas

El registro conserva la fecha, plataforma, consulta exacta, filtros, número de resultados y archivo exportado. El archivo de trabajo se crea solo si todavía no existe.

In [ ]:
template_path = ROOT / 'data' / 'templates' / 'search_log.csv'
working_path = ROOT / 'data' / 'interim' / 'search_log_working.csv'
working_path.parent.mkdir(parents=True, exist_ok=True)

if not working_path.exists():
    shutil.copyfile(template_path, working_path)
    print(f'Created: {working_path.relative_to(ROOT)}')
else:
    print(f'Already exists: {working_path.relative_to(ROOT)}')

search_log = pd.read_csv(working_path)
search_log.head()

## 5. Preparar una fila candidata

Esta celda no escribe automáticamente. Completa la información después de ejecutar una búsqueda real.

In [ ]:
candidate_search = {
    'search_id': 'YYYYMMDD_platform_language_querytype',
    'search_date': '',
    'reviewer': '',
    'source_channel': 'scientific_database',
    'platform_or_website': '',
    'database_collection': '',
    'language': selected_language,
    'query_type': selected_type,
    'exact_query': selected['exact_query'],
    'filters_applied': '',
    'date_from': strategy['eligibility']['date_from'],
    'date_to': strategy['eligibility']['date_to'],
    'result_count': None,
    'export_filename': '',
    'export_format': '',
    'search_status': 'planned',
    'notes': '',
}

pd.DataFrame([candidate_search])

## 6. Fuentes prioritarias

In [ ]:
scientific = pd.DataFrame(strategy['search_channels']['scientific_databases'])
institutional = pd.DataFrame({
    'institutional_source': strategy['search_channels']['institutional_sources']
})

display(scientific)
display(institutional)

## Criterio para avanzar

Antes de importar resultados, deben cumplirse estos puntos:

- consultas revisadas en los cuatro idiomas;
- plataformas prioritarias definidas;
- filtros y periodo documentados;
- conjunto de artículos conocidos para comprobar sensibilidad;
- registro de búsqueda preparado;
- prueba piloto antes de realizar la búsqueda completa.